To log in to Hugging Face programmatically, you can use the `login` function from the `huggingface_hub` library. This function will use your `HF_TOKEN` from Colab secrets if it's set.

In [ ]:
from huggingface_hub import login
from google.colab import userdata

# Retrieve the HF_TOKEN from Colab secrets
hf_token = userdata.get('Token')

# Log in to Hugging Face
if hf_token:
    login(token=hf_token)
    print("Successfully logged in to Hugging Face.")
else:
    print("HF_TOKEN not found in Colab secrets. Please set it to access gated datasets.")

In [ ]:
# Install dependencies
!pip -q install datasets

from datasets import load_dataset
import pandas as pd
from google.colab import userdata

# Ensure hf_token is available; retrieve it if not already defined
try:
    hf_token
except NameError:
    print("HF_TOKEN variable not found, attempting to retrieve from Colab secrets.")
    hf_token = userdata.get('Token')
    if not hf_token:
        print("HF_TOKEN not found in Colab secrets. Please ensure it's set.")


# ---------- WildChat ----------
print("=" * 80)
print("LOADING WILDCHAT")
print("=" * 80)

wildchat = load_dataset(
    "allenai/WildChat",
    split="train",
    token=hf_token # Explicitly pass the token for authentication
)

print("Total rows:", len(wildchat))
print("\nSample rows:\n")

for i in range(3):
    print(f"\n--- SAMPLE {i+1} ---")
    print(wildchat[i])
    print()

safe_wildchat = wildchat.filter(lambda x: x["toxic"] == False)


# ---------- LMSYS Chat 1M ----------
print("\n" + "=" * 80)
print("LOADING LMSYS-CHAT-1M")
print("=" * 80)

lmsys = load_dataset(
    "lmsys/lmsys-chat-1m",
    split="train",
    token=hf_token # Explicitly pass the token for authentication
)

print("Total rows:", len(lmsys))
print("\nSample rows:\n")

for i in range(3):
    print(f"\n--- SAMPLE {i+1} ---")
    print(lmsys[i])
    print()

In [ ]:
from datasets import load_dataset

jb = load_dataset(
    "TrustAIRLab/in-the-wild-jailbreak-prompts",
    name="jailbreak_2023_05_07",  # Added config name
    split="train"
)

print("Rows:", len(jb))

for i in range(3):
    print(jb[i])
    print()

In [ ]:
from datasets import load_dataset
from google.colab import userdata

# Ensure hf_token is available; retrieve it if not already defined (assuming 0f23a5aa was run)
try:
    hf_token
except NameError:
    print("HF_TOKEN variable not found, attempting to retrieve from Colab secrets.")
    hf_token = userdata.get('Token')
    if not hf_token:
        print("HF_TOKEN not found in Colab secrets. Please ensure it's set.")

hack = load_dataset(
    "hackaprompt/hackaprompt-dataset",
    split="train",
    token=hf_token # Explicitly pass the token for authentication
)

print("Rows:", len(hack))

for i in range(3):
    print(hack[i])
    print()

After successfully logging in, you can try loading the `lmsys/lmsys-chat-1m` dataset again. Remember to run the cell containing `load_dataset("lmsys/lmsys-chat-1m", split="train")` after executing the login cell.

In [ ]:
# BUCKET CONSTRUCTION

# Bucket A — Synthetic Safe
# from your existing combined_dataset_final.csv
# filter label == 0 (safe), source == WildJailbreak benign
# sample 500

# Bucket B — Synthetic Unsafe
# filter label == 1 (unsafe), source == WildJailbreak harmful
# sample 500

# Bucket C — Human Safe (WildChat)
# extract conversation[0]['content'] where role == 'user'
# filter toxic == False AND language == 'English'
# sample 500

# Bucket D — Human Unsafe (TrustAIRLab)
# use full 666 rows, extract 'prompt' field
# all jailbreak == True already

# Bucket E — Control (AdvBench)
# use all 520, extract 'prompt' field
# this is your "easy unsafe" sanity check

In [ ]:
import pandas as pd
df = pd.read_csv('compressed_data.csv.gz')
print(df.columns.tolist())
print(df.head(3))
print(df['source'].value_counts() if 'source' in df.columns else "no source column")

In [ ]:
print(df[df['source'] == 'wildjailbreak']['label'].value_counts())

In [ ]:
import pandas as pd
from datasets import load_dataset
import random
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.linear_model import LogisticRegression

random.seed(42)

# ── load your existing dataset ──────────────────────────────────────────
df = pd.read_csv('compressed_data.csv.gz')

# ── BUCKET A — Synthetic Safe ───────────────────────────────────────────
bucket_a = df[(df['source'] == 'wildjailbreak') & (df['label'] == 'safe')].sample(500, random_state=42)[['prompt']].copy()
bucket_a['bucket'] = 'A_synthetic_safe'
bucket_a['true_label'] = 'safe'

# ── BUCKET B — Synthetic Unsafe ─────────────────────────────────────────
bucket_b = df[(df['source'] == 'wildjailbreak') & (df['label'] == 'unsafe')].sample(500, random_state=42)[['prompt']].copy()
bucket_b['bucket'] = 'B_synthetic_unsafe'
bucket_b['true_label'] = 'unsafe'

# ── BUCKET C — Human Safe (WildChat streamed) ───────────────────────────
wildchat = load_dataset("allenai/WildChat", split="train", streaming=True, token=hf_token)

collected = []
for row in wildchat:
    if len(collected) >= 500:
        break
    if (
        row['toxic'] == False and
        row['language'] == 'English' and
        row['conversation'] and
        row['conversation'][0]['role'] == 'user'
    ):
        collected.append(row['conversation'][0]['content'])

bucket_c = pd.DataFrame({'prompt': collected, 'bucket': 'C_human_safe', 'true_label': 'safe'})

# ── BUCKET D — Human Unsafe (TrustAIRLab) ───────────────────────────────
bucket_d = df[(df['source'] == 'trustairlab') & (df['label'] == 'unsafe')][['prompt']].copy()
bucket_d['bucket'] = 'D_human_unsafe'
bucket_d['true_label'] = 'unsafe'

# ── combine ──────────────────────────────────────────────────────────────
nb6_df = pd.concat([bucket_a, bucket_b, bucket_c, bucket_d], ignore_index=True)

print("Bucket sizes:")
print(nb6_df['bucket'].value_counts())
print("\nTotal rows:", len(nb6_df))

# ── TRAIN VECTORIZER AND CLASSIFIER ──────────────────────────────────────
# Using the combined dataset for demonstration. In a real scenario, you'd want
# a separate training set.

X_train = nb6_df['prompt']
y_train = nb6_df['true_label']

vectorizer = TfidfVectorizer()
X_train_vectorized = vectorizer.fit_transform(X_train)

clf = LogisticRegression(random_state=42, solver='liblinear') # Using liblinear for smaller datasets/binary classification
clf.fit(X_train_vectorized, y_train)

print("\nVectorizer and Classifier trained successfully.")

# ── STEP 5: qualitative analysis ─────────────────────────────────────────
fp_examples = nb6_df[nb6_df['bucket'] == 'C_human_safe'].copy()
fp_examples['predicted'] = clf.predict(vectorizer.transform(fp_examples['prompt']))
fps = fp_examples[fp_examples['predicted'] == 'unsafe']['prompt'].head(5).tolist()

print("── FALSE POSITIVES (benign flagged as unsafe) ──")
for i, p in enumerate(fps):
    print(f"\n{i+1}. {p[:300]}")

fn_examples = nb6_df[nb6_df['bucket'] == 'D_human_unsafe'].copy()
fn_examples['predicted'] = clf.predict(vectorizer.transform(fn_examples['prompt']))
fns = fn_examples[fn_examples['predicted'] == 'safe']['prompt'].head(5).tolist()

print("\n── FALSE NEGATIVES (jailbreaks missed) ──")
for i, p in enumerate(fns):
    print(f"\n{i+1}. {p[:300]}")

In [ ]:
# ── TRAIN ON FULL CORPUS (excluding TrustAIRLab) ──────────────────────
# Do NOT train on the bucket data itself

# Load full training corpus
df_full = pd.read_csv('compressed_data.csv.gz')
df_full = df_full[df_full['source'] != 'trustairlab'].reset_index(drop=True)

# Train vectorizer and classifier on full corpus
vectorizer = TfidfVectorizer(max_features=10000, ngram_range=(1, 2))
X_full_train = vectorizer.fit_transform(df_full['prompt'])
y_full_train = df_full['label']

clf = LogisticRegression(max_iter=1000, class_weight='balanced')
clf.fit(X_full_train, y_full_train)

print(f"Classifier trained on {len(df_full):,} prompts (full corpus, no TrustAIRLab)")
print(f"Vocabulary size: {len(vectorizer.vocabulary_):,}")
print()

# ── EVALUATE ON BUCKETS (held-out test set) ──────────────────────────
# Now predict on the bucket data

predictions = {}
for bucket_name in ['A_synthetic_safe', 'B_synthetic_unsafe', 'C_human_safe', 'D_human_unsafe']:
    bucket_data = nb6_df[nb6_df['bucket'] == bucket_name].copy()
    X_bucket = vectorizer.transform(bucket_data['prompt'])
    bucket_data['pred'] = clf.predict(X_bucket)
    predictions[bucket_name] = bucket_data

# Print metrics per bucket
from sklearn.metrics import confusion_matrix, recall_score, precision_score

print("=" * 70)
print("BUCKET-BY-BUCKET EVALUATION (train on 110k, test on buckets)")
print("=" * 70)
print()

for bucket_name in ['A_synthetic_safe', 'B_synthetic_unsafe', 'C_human_safe', 'D_human_unsafe']:
    bucket_df = predictions[bucket_name]
    y_true = bucket_df['true_label']
    y_pred = bucket_df['pred']

    if bucket_name.endswith('_safe'):
        # For safe buckets, report FP rate
        fp_rate = (y_pred == 'unsafe').sum() / len(y_pred)
        print(f"{bucket_name:20} | FP rate: {fp_rate:.3f} (n={len(bucket_df)})")
    else:
        # For unsafe buckets, report recall
        recall = recall_score(y_true, y_pred, pos_label='unsafe', zero_division=0)
        print(f"{bucket_name:20} | Recall: {recall:.3f} (n={len(bucket_df)})")

print()

In [ ]:
# false positives — benign WildChat prompts flagged as unsafe
fp_examples = nb6_df[
    (nb6_df['bucket'] == 'C_human_safe')
].copy()
fp_examples['predicted'] = clf.predict(vectorizer.transform(fp_examples['prompt']))
fps = fp_examples[fp_examples['predicted'] == 'unsafe']['prompt'].head(5).tolist()

print("── FALSE POSITIVES (benign flagged as unsafe) ──")
for i, p in enumerate(fps):
    print(f"\n{i+1}. {p[:300]}")

# false negatives — real jailbreaks missed by classifier
fn_examples = nb6_df[
    (nb6_df['bucket'] == 'D_human_unsafe')
].copy()
fn_examples['predicted'] = clf.predict(vectorizer.transform(fn_examples['prompt']))
fns = fn_examples[fn_examples['predicted'] == 'safe']['prompt'].head(5).tolist()

print("\n── FALSE NEGATIVES (jailbreaks missed) ──")
for i, p in enumerate(fns):
    print(f"\n{i+1}. {p[:300]}")

## Qualitative Analysis

### False Positives (Human Safe → Predicted Unsafe)
The classifier consistently misclassifies creative writing requests as unsafe.
Film scripts, character descriptions, and narrative prompts share surface-level
structure with adversarial jailbreaks — elaborate setup, fictional framing,
detailed instructions. Since the training data associates this structure with
malicious intent, the classifier over-generalises to all creative content.

### False Negatives (Human Unsafe → Predicted Safe)
The missed jailbreaks contain no harmful vocabulary. Several read as completely
ordinary questions ("hello", "how to earn from amazon"). The malicious intent
is entirely concealed within context, persona, or prior conversation setup —
invisible to a bag-of-words model.

### Implication
The classifier has learned a surface pattern (elaborate framing = unsafe)
rather than semantic intent. This explains both failure modes simultaneously:
it fires on harmless creative content and stays silent on intent-concealed attacks.